In [14]:
import requests
import time
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup


article_list = []
base_url = "https://siryc.or.kr"
list_url = f"{base_url}/boards/news"

for page in range(1, 3):
    time.sleep(1)  # Be polite and avoid overwhelming the server
    response = requests.get(list_url, params={"page": page})
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    # 게시글은 turbo-frame으로 lazy-load 되므로,
    # 각 프레임의 src(/posts/{id}/col_x)를 별도로 요청해야 한다.
    frames = soup.select('#frame-posts turbo-frame[id^="col-x-post-grid_post_"]')
    for frame in frames:
        src = frame.get("src")
        if not src:
            continue

        time.sleep(0.3)
        post_res = requests.get(urljoin(base_url, src))
        post_res.encoding = "utf-8"
        post_soup = BeautifulSoup(post_res.text, "html.parser")

        anchor = post_soup.select_one('a[href*="/posts/"]')
        if anchor is None:
            continue

        title_el = anchor.select_one("h5")
        # anchor 내부에 p.text-gray-500이 두 개(설명 / 날짜) 있으므로
        # footer flex 박스 안에 있는 것을 명시적으로 골라준다.
        date_el = anchor.select_one("div.justify-between p.text-gray-500")

        article_list.append({
            "title": title_el.get_text(strip=True) if title_el else "",
            "link": anchor["href"],
            "date": date_el.get_text(strip=True) if date_el else "",
        })

df = pd.DataFrame(article_list, columns=["title", "date", "link"])
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date
df = df.sort_values("date", ascending=False, na_position="last").reset_index(drop=True)
df.index = df.index + 1
df.index.name = "no"

pd.set_option("display.max_colwidth", None)

styled = (
    df.style
      .format({"link": lambda u: f'<a href="{u}" target="_blank">바로가기</a>'})
      .set_properties(**{"text-align": "left", "vertical-align": "top"})
      .set_table_styles([
          {"selector": "th", "props": [("text-align", "left"), ("background-color", "#f5f5f5")]},
          {"selector": "td", "props": [("padding", "6px 10px")]},
      ])
)
styled


,title,date,link
no,,,
1,"안녕, 기지개 Vol.2 ② 직원 인터뷰 ㅣ 여유를 찾아 분주하게 움직이는 사람, '배성민' 팀장 <요즘, 뭐하고 지내요?>",2026-05-26,바로가기
2,"안녕, 기지개 Vol.2 ② 직원 인터뷰 ㅣ 누군가의 일상에 초록을 더하는 사람, '윤상호' 매니저 <요즘, 뭐하고 지내요?>",2026-05-26,바로가기
3,"안녕, 기지개 Vol.2 ① 청년 인터뷰 ㅣ 따뜻하고 감성적인 사람, '와비사비' <요즘, 뭐하고 지내요?>",2026-05-26,바로가기
4,리필스테이션 운영 변경 안내(기지개컴퍼니 청년 기획 프로그램),2026-05-22,바로가기
5,[서울청년기지개센터 × 서울지방우정청] 고립예방우편 서비스 신청,2026-05-22,바로가기
6,우리집 공간 프로그램 '6월의 찾아가는 정원처방',2026-05-21,바로가기
7,[모집 안내] 기지개 시즌즈(전통무예 체험 프로그램-6월) 참가자 모집 안내,2026-05-21,바로가기
8,"[다시, 한 걸음] 사회기술훈련 1기 프로그램 후기",2026-05-19,바로가기
9,5월 사회진입준비도 특강 참여자 모집,2026-05-18,바로가기


In [13]:
pip install jinja2


   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   ---------------------------------------- 2/2 [jinja2]

Note: you may need to restart the kernel to use updated packages.
